# Using Built-in Tools in LangChain: Search and Wikipedia : Here we will see build in tools in langchain spefically tools like DuckduckGo search and wikipedia tools, unlike the our calcluator tool, this tool connect to external knowledge sources allowing the agent to fetch uptodate facts and backgroud informations .

Key take away is: Agent do need to only relay on what is inside the model, they can extend their knowledge by calling search and reference tools on Demand 

In [1]:
# Install packages
!pip install -qU langchain langchain-google-genai google-generativeai==0.8.5


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -q ddgs


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=ed573683bf3b7eef0b9d50ae60cac6a5c1dc95f0ac4121d1d227cd2428fc6ac1
  Stored in directory: c:\users\sileymane.dianka\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\8f\ab\cb\45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia

   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [wikipedia]




[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
#  Set your Gemini API key\n AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"
):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print(os.environ.get("GOOGLE_API_KEY"))

AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg


# Define tools (DuckDuckGo + Wikipedia)

In [7]:
from typing import List
from ddgs import DDGS
import wikipedia

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent 

wikipedia.set_lang("en")

def _format_hits(hits: List[dict], k: int = 5) -> str:
    lines = []
    for i, h in enumerate(hits[:k], 1):
        title = h.get("title") or h.get("heading") or ""
        url = h.get("href") or h.get("url") or ""
        snippet = h.get("body") or h.get("description") or h.get("snippet") or ""
        lines.append(f"{i}. {title}\n   {url}\n   {snippet}")
    return "\n".join(lines) if lines else "No results."

@tool
def web_search(query: str, max_results: int = 5, time_range: str = None) -> str:
    """DuckDuckGo web search. Returns up to `max_results` compact results with URLs and snippets.
    time_range can be 'd','w','m','y' (day, week, month, year) for recency, or None."""
    try:
        with DDGS() as ddg:
            hits = list(ddg.text(query, max_results=max_results, timelimit=time_range))
        return _format_hits(hits, k=max_results)
    except Exception as e:
        return f"Search error: {e}"

@tool
def wiki_summary(topic: str, sentences: int = 3) -> str:
    """Return a short Wikipedia summary with the canonical page URL."""
    try:
        page = wikipedia.page(topic, auto_suggest=True, redirect=True)
        summary = wikipedia.summary(page.title, sentences=sentences)
        return f"Title: {page.title}\nURL: {page.url}\nSummary: {summary}"
    except wikipedia.DisambiguationError as e:
        opts = "; ".join(e.options[:5])
        return f"Disambiguation: be specific. Options include: {opts}"
    except wikipedia.PageError:
        return "No matching page found."
    except Exception as e:
        return f"Wikipedia error: {e}"

# Build the search agent

In [8]:
def build_search_agent():
    llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)
    tools = [web_search, wiki_summary]

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research assistant.\n"
         "Use tools for ANY factual claims, numbers, dates, or 'latest' info.\n"
         "Choose:\n"
         "- web_search: for fresh/news or when URLs are required.\n"
         "- wiki_summary: for background/definitions.\n"
         "When you cite, include page titles and URLs from tool outputs.\n"
         "Keep answers concise. Prefer top 3 results. Do not hallucinate."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])

    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=False)  # clean output

search_agent = build_search_agent()
print(" Search agent ready.")

 Search agent ready.


# Quick tests

In [9]:
def ask(q: str):
    res = search_agent.invoke({"input": q})
    print(res["output"], "\n")

# 1) Simple fact + citation
ask("Who won the FIFA World Cup 2022? Give one sentence and include a source URL.")

# 2) Background from Wikipedia
ask("In 3 bullets, explain Retrieval-Augmented Generation. Cite the Wikipedia page with URL.")

# 3) Fresh headlines (limit and summarize)
ask("Find 3 recent headlines about generative AI this week. One line per item with URL.")

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 43.759704092s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '